In [8]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# -------------------------------------------------------------
# 1. Timing & Kinematics Configuration (Total: 180 frames ~ 12 sec)
# -------------------------------------------------------------
n_phase1 = 45   # Phase 1: Particle at rest, NO field outside (0 - 3s)
n_phase2 = 35   # Phase 2: Magnetic field appears outside the box (3s - 5.3s)
n_phase3 = 100  # Phase 3: Particle slowly moves upward (5.3s - 12s)
n_total = n_phase1 + n_phase2 + n_phase3

y_start = -0.6
y_target = 0.75

# Precompute particle vertical position
y_pos = np.full(n_total, y_start)
t_up = np.linspace(0, 1, n_phase3)
# Very slow and smooth acceleration upward
y_pos[n_phase1 + n_phase2:] = y_start + (y_target - y_start) * (t_up ** 1.5)

# Precompute external field fade-in alpha
field_alpha = np.zeros(n_total)
field_alpha[n_phase1:n_phase1 + n_phase2] = np.linspace(0.0, 0.45, n_phase2)
field_alpha[n_phase1 + n_phase2:] = 0.45

# -------------------------------------------------------------
# 2. Figure Setup (Ultra-clean, Minimalist)
# -------------------------------------------------------------
fig, ax = plt.subplots(figsize=(6.5, 5.0), dpi=90)

ax.set_xlim(-1.8, 1.8)
ax.set_ylim(-1.3, 1.5)
ax.set_aspect('equal')
ax.grid(True, linestyle='--', alpha=0.2)

ax.set_title("Frame S': View from Inside the Moving Box", fontsize=11, pad=12)
ax.set_xlabel("x'", fontsize=10)
ax.set_ylabel("y'", fontsize=10)

# Geometry of the Box
box_w, box_h = 2.2, 2.0
box_x_min, box_x_max = -box_w/2, box_w/2
box_y_min, box_y_max = -0.9, -0.9 + box_h

box_patch = Rectangle((box_x_min, box_y_min), box_w, box_h,
                      edgecolor='#f59e0b', facecolor='#f59e0b', alpha=0.07, lw=2.5)
ax.add_patch(box_patch)

# Observer inside the box (at rest on the left)
ax.plot([-0.7], [-0.35], 'o', color='#38bdf8', markersize=9)
ax.text(-0.7, -0.15, "Observer (S')", color='#38bdf8', fontsize=8.5, ha='center')

# External Magnetic Field Grid (Strictly positioned OUTSIDE the box)
gx, gy = np.meshgrid(np.linspace(-1.7, 1.7, 9), np.linspace(-1.2, 1.4, 8))
gx, gy = gx.flatten(), gy.flatten()
outside_mask = (gx < box_x_min - 0.12) | (gx > box_x_max + 0.12) | \
               (gy < box_y_min - 0.12) | (gy > box_y_max + 0.12)
bx_pts, by_pts = gx[outside_mask], gy[outside_mask]

# Field plot elements (initially invisible with alpha=0)
b_circles = ax.scatter(bx_pts, by_pts, marker='o', s=110, facecolors='none', 
                       edgecolors='#0284c7', alpha=0.0, lw=1.2)
b_crosses = ax.scatter(bx_pts, by_pts, marker='x', s=40, 
                       color='#0284c7', alpha=0.0, lw=1.2)
b_label = ax.text(1.65, 1.35, r"$\mathbf{B}_{\mathrm{ext}}$", color='#0284c7', 
                  fontsize=10, ha='right', weight='bold', alpha=0.0)

# Moving Particle & Vertical Trajectory Trail
trail, = ax.plot([], [], color='#ef4444', lw=2.2, alpha=0.55)
particle, = ax.plot([], [], 'o', color='#ef4444', markersize=9)

# -------------------------------------------------------------
# 3. Animation Update Function
# -------------------------------------------------------------
start_move_idx = n_phase1 + n_phase2

def update(i):
    # 1. Update external field visibility
    curr_alpha = field_alpha[i]
    b_circles.set_alpha(curr_alpha)
    b_crosses.set_alpha(curr_alpha)
    b_label.set_alpha(min(1.0, curr_alpha * 2.2))
    
    # 2. Update particle position
    curr_y = y_pos[i]
    particle.set_data([0], [curr_y])
    
    # 3. Update trajectory trail (only during Phase 3)
    if i >= start_move_idx:
        trail_y = y_pos[start_move_idx:i+1]
        trail_x = np.zeros_like(trail_y)
        trail.set_data(trail_x, trail_y)
    else:
        trail.set_data([], [])
        
    return b_circles, b_crosses, b_label, particle, trail

# -------------------------------------------------------------
# 4. Render and Save (15 fps -> ~12 seconds of smooth slow-motion)
# -------------------------------------------------------------
anim = FuncAnimation(fig, update, frames=n_total, interval=65, blit=True)

anim.save('box_inside_view.mp4', writer='ffmpeg', fps=15, extra_args=['-vcodec', 'libx264'])
plt.close(fig)

HTML(anim.to_html5_video())